## Loading data

In [1]:
# generating synthetic spectra with different noise levels
import pandas as pd
import numpy as np
import kennard_stone as ks

# Add parent directory to sys.path so local module 'synthetic' (one level up) can be imported
import sys
from pathlib import Path # for path manipulations
parent_dir = Path.cwd().parent.parent.resolve() # move two levels up from current working directory
if str(parent_dir) not in sys.path: # check to avoid duplicates
    sys.path.insert(0, str(parent_dir)) # insert at the start of sys.path to prioritize local modules

from synthetic import generate_synthetic_spectral_data

config = [
    {
        'nome': 'A',
        'n_amostras': 156,
        'picos': [250, 380, 550, 700, 850],  # 4 picos
        'amp_media': 1.0,
        'amp_std': 0.3,
        'larg_media': 15.0,
        'larg_std': 2.0,
        'ruido_std': 0.04
    },
    {
        'nome': 'B',
        'n_amostras': 146,
        'picos': [50, 250, 380, 550, 850],  # 3 picos (sem pico em 550)
        'amp_media': 1.4,
        'amp_std': 0.5,
        'larg_media': 15.0,
        'larg_std': 1.8,
        'ruido_std': 0.035
    }
]

data_complete = generate_synthetic_spectral_data(
    configuracao_classes=config,
    n_pontos=500,
    x_min=1,
    x_max=1000,
    seed=0
)

import pandas as pd
pd.options.plotting.backend = 'plotly' # setting plotly as the backend for pandas plotting 
data_complete.iloc[:, 1:].T.plot()

## PLS- (R or DA) modeling

In [2]:
# Split dataset by class and create calibration/prediction sets using Kennard-Stone (as in original pipeline)
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.iloc[:, 1:], test_size=0.30)  # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.iloc[:, 1:], test_size=0.30)  # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True)  # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0])  # target for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0])  # target for prediction set

# Xcalclass_prep = Xcalclass.copy()
# Xpredclass_prep = Xpredclass.copy()

# preprocessings
import preprocessings as prepr  # preprocessing methods
Xcalclass_prep, mean_calclass  = prepr.mc(Xcalclass)
Xpredclass_prep = Xpredclass - mean_calclass

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:41:37,084 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:41:37,103 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:41:37,133 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 

In [3]:
# PLS-DA with optimized latent variables
from modeling import pls_optimized

plsda_results = pls_optimized(
    Xcalclass_prep, 
    ycalclass,
    LVmax=1,
    Xpred=Xpredclass_prep,
    ypred=ypredclass,
    aim='classification',
    cv=10
)

# Convenience references used later
pls_model = plsda_results[3]               # fitted PLS model
vip_scores_mat = plsda_results[4]          # VIP scores matrix (features × LV)
y_pred_cont = plsda_results[5].iloc[:, -1] # continuous predictions for Xcalclass (used for MI/Cov)

# plotando o vip scores rapidamente
vip_scores_mat.T.plot()

## Spectral cuts (domain knowledge)

In [5]:
# establishing spectral cuts based on expert knowledge of XRF spectra
spectral_cuts = [
('F1', 1.0, 100.0),
('background1', 100.0, 200.0),
('F2', 200.0, 300.0),
('background2', 300.0, 330.0),
('F3', 330.0, 430.0),
('background3', 430.0, 500.0),
('F4', 500.0, 600.0),
('background4', 600.0, 660.0),
('F5', 660.0, 750.0),
('background5', 750.0, 815.0),
('F6', 815.0, 890.0),
('background6', 890.0, 1000.0)
]

import explaining as exp
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_sums_df = exp.aggregate_spectral_zones(spectral_zones_class, aggregator='extreme')
predicates_quantiles = exp.predicates_by_quantiles(zone_sums_df, [0.2, 0.4, 0.6, 0.8])
co_occurrence_matrix_df = predicates_quantiles[2]
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_sums_df,
    y_predicted_numeric=y_pred_cont
)

## VIP, Regression Coefficients e SHAP (como no original)

In [6]:
# VIP scores por energia
vip_scores_df = pd.DataFrame({
    'energy': vip_scores_mat.T.index,
    'VIP_Score': vip_scores_mat.T.iloc[:,0].values
})
vip_scores_df = vip_scores_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in vip_scores_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
vip_scores_df['Zone'] = vip_scores_df['energy'].map(energy_to_zone_vip)
vip_scores_unique_df = vip_scores_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
vip_scores_unique_df = vip_scores_unique_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# Coeficientes de regressão do PLS
reg_vet = pd.DataFrame(pls_model.coef_, columns=pls_model.feature_names_in_).T
reg_vet.insert(0, 'energy', reg_vet.index)
reg_vet = reg_vet.reset_index(drop=True)
reg_vet.columns = ['energy','Reg_coef']
reg_vet['Abs_Reg_coef'] = reg_vet['Reg_coef'].abs()
reg_vet = reg_vet.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)
energy_to_zone_reg = {}
for zone_name, start, end in spectral_cuts:
    for e in reg_vet['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_reg[e] = zone_name
reg_vet['Zone'] = reg_vet['energy'].map(energy_to_zone_reg)
reg_vet_unique_df = reg_vet.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
reg_vet_unique_df = reg_vet_unique_df.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)

# # vamos agora extrair as variaveis mais importantes atraves do método SHAP
# import shap

# # Para PLSRegression, usamos KernelExplainer porque não há explainer dedicado muito rápido
# explainer_pls = shap.KernelExplainer(plsda_results[3].predict, Xcalclass_prep)
# shap_values_pls = explainer_pls(Xcalclass_prep)

# shap_global_importance = pd.DataFrame({
#     'energy': Xpredclass_prep.columns,
#     'Mean_Abs_SHAP': np.abs(shap_values_pls.values).mean(axis=0)}) # tomando a importancia global como a media dos valores absolutos dos valores SHAP para cada feature
# shap_global_importance.sort_values(by='Mean_Abs_SHAP', ascending=False, inplace=True)

# # vamos gerar uma nova coluna em shap_global_importance com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
# energy_to_zone_shap = {}
# for zone_name, start, end in spectral_cuts:
#     for i in shap_global_importance['energy']:
#         i_float = float(i)
#         if start <= i_float <= end:
#             energy_to_zone_shap[i] = zone_name
# shap_global_importance['Zone'] = shap_global_importance['energy'].map(energy_to_zone_shap)

# # agora vamos filtrar shap_global_importance para manter apenas as zonas espectrais únicas com maior SHAP score
# shap_unique_df = shap_global_importance.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
# shap_unique_df = shap_unique_df.sort_values(by='Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
# shap_unique_df.to_csv('shap_synthetic.csv', index=False, sep=';')
shap_unique_df = pd.read_csv('shap_synthetic.csv', sep=';') # loading previously saved shap_unique_df

# **Comparando com o bagging**

In [7]:
# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 42]

all_results = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_predicted_numeric = plsda_results[5].iloc[:, -1] # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_predicted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=20,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    # Calcular MI
    mi_results_dict_seed = exp.calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance',
        threshold=0.001, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    # Salvar no dicionário principal
    all_results[seed] = {
        'bags_result': bags_result_seed,
        'mi_results_dict': mi_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")

# Construir grafo para esta semente
    DG = exp.build_predicate_graph(
        bags_result=all_results[seed]['bags_result'],
        mi_results_dict=all_results[seed]['mi_results_dict'],
        co_occurrence_matrix_df=co_occurrence_matrix_df,
        predicates_df=predicates_quantiles[0],
        random_state=seed,
        show_details=True
    )

#     DG = exp.build_fold_predicate_graph(
#     bags_result=all_results[seed]['bags_result'],
#     mi_results_dict=all_results[seed]['mi_results_dict'],
#     predicates_df=predicates_quantiles[0],
#     random_state=42,
#     show_details=True,
#     normalize_weights=True,
#     weight_mode='ranking',
#     co_occurrence_matrix=co_occurrence_matrix_df,
#     apply_confidence_multiplier=True,
#     accumulate_cooccurrence_weights=True
# )

    # Armazenar grafo
    graphs_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    lrc_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_by_seed[seed] = lrc_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_df_seed = lrc_by_seed[seed].rename(columns={'Node': f'Predicate_Seed_{seed}'})
    lrc_all_seeds_df = pd.concat([lrc_all_seeds_df, lrc_df_seed[[f'Predicate_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_unique_by_seed = {}
for seed, lrc_df in lrc_by_seed.items():
    lrc_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_unique_df = lrc_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_unique_by_seed[seed] = lrc_unique_df

lrc_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 73 | Descartados: 23
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 74 | Descartados: 22
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 74 | Descartados: 22
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 75 | Descartados: 21
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 73 | Descartados: 23
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 74 | Descartados: 22
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 73 | Descartados: 23
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 73 | Descartados: 23
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 74 | Descartados: 22
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 74 | Descartados: 22
Bag_11 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 73 | Descartados: 23
Bag_12 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 73 | Descartados: 23
Bag_13 | Amo

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide




Processando LRC do grafo...

Processando LRC do grafo...


C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide



,Predicate_Seed_0,Predicate_Seed_1,Predicate_Seed_42
0,F1 > -0.71,F1 > -0.71,F1 > -0.71
1,F1 > -0.69,F1 > -0.69,F1 > -0.69
2,F5 > -0.54,F5 > -0.54,F5 > -0.54
3,F1 <= 0.83,F1 <= 0.83,F1 <= 0.83
4,F5 <= 0.61,F5 <= 0.61,F5 <= 0.61
...,...,...,...
71,background1 <= -0.08,background2 <= -0.08,background2 > 0.07
72,background6 <= -0.10,background6 <= -0.10,background4 <= -0.09
73,background2 > 0.07,background4 <= -0.09,Class_A
74,Class_A,Class_A,Class_B


# Kennard-Stone + Round-Robin k-fold

## Duas Estratégias Disponíveis

### 1. Estratégia GLOBAL (`per_predicate=False`) - Original
- KS é aplicado **globalmente** em todas as amostras do dataset
- Distribui amostras via round-robin para k folds
- **Todos os predicados compartilham os mesmos folds**
- Predicados com cobertura < min_samples são eliminados

**Vantagens:**
- Consistência: mesmas amostras nos mesmos folds para todos os predicados
- Comparabilidade direta entre predicados
- Menor custo computacional (KS executado uma única vez)

**Desvantagens:**
- Predicados com baixa cobertura global podem ser eliminados
- A diversidade do KS é otimizada globalmente, não por predicado

---

### 2. Estratégia PER-PREDICATE (`per_predicate=True`) - Nova
- KS é aplicado **individualmente** para cada predicado
- Considera apenas as amostras que satisfazem cada predicado
- **Cada predicado tem seus próprios folds independentes**
- Resultados são combinados ao final

**Vantagens:**
- Maximiza representatividade dentro de cada predicado
- Mais amostras válidas por predicado (menos eliminações)
- Independência estatística entre predicados
- Diversidade otimizada para cada predicado individualmente

**Desvantagens:**
- Folds inconsistentes entre predicados (amostra X pode estar no Fold_1 para um predicado e Fold_3 para outro)
- Maior custo computacional (KS executado N vezes)
- Combinação de resultados requer cuidado na interpretação

**Solução técnica para KS unidimensional:**
- KS precisa de múltiplas variáveis para calcular distâncias
- Solução: adicionar o índice normalizado da amostra como segunda coluna
- Isso é determinístico e representa a "posição temporal" da amostra

In [8]:
import ks_folding as ksf

folds_result = ksf.kfold_predicates_roundrobin(
    zone_sums_df=zone_sums_df,
    y_predicted_numeric=y_pred_cont,
    predicates_df=predicates_quantiles[0],
    k_folds=2,
    min_samples_ratio=0.001,  # 60% das amostras do fold
    verbose=True,
    per_predicate=True # escolhe entre fazer o fold por predicado ou globalmente (que faz o fold para todos os predicados juntos)
)

# Adiciona classe prevista (A/B) em cada DataFrame de predicado
for fold_name, pred_dict in folds_result.items():
    for rule, df_info in pred_dict.items():
        df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B')

mi_results_dict = exp.calculate_predicate_metrics(
    bags_result=folds_result,
    metric='covariance',
    threshold=0.001,
    #n_neighbors=5
)

max_len = max(len(mi_df['Predicate']) for mi_df in mi_results_dict.values())
padded_dict = {
    f'Predicate_{fold}': list(mi_df['Predicate']) + [None]*(max_len - len(mi_df['Predicate']))
    for fold, mi_df in mi_results_dict.items()
}
all_cov_results = pd.DataFrame(padded_dict)
all_cov_results

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:45:39,838 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:45:39,839 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 

Configuração KS + Round-Robin
Estratégia: PER-PREDICATE (individual)
Total de amostras: 211
Número de folds: 2
Amostras por fold (aprox.): 105
Mínimo de amostras por predicado: 2 (0% do fold)

Iniciando estratégia PER-PREDICATE...
KS será aplicado individualmente para cada predicado.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:45:40,062 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:45:40,067 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 96
Predicados válidos: 96
Predicados eliminados: 0
Folds criados: 2

Estatísticas por predicado válido:
  'F1 <= -0.71': 46 amostras, folds: [23, 23]
  'F1 > -0.71': 165 amostras, folds: [83, 82]
  'F1 <= -0.69': 79 amostras, folds: [40, 39]
  'F1 > -0.69': 132 amostras, folds: [66, 66]
  'F1 <= 0.22': 126 amostras, folds: [63, 63]
  ... e mais 91 predicados

Predicados por fold:
  Fold_1: 96 predicados
  Fold_2: 96 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001


,Predicate_Fold_1,Predicate_Fold_2
0,F1 > -0.71,F1 > -0.71
1,F1 > -0.69,F1 > -0.69
2,F1 <= 0.83,F5 > -0.54
3,F5 > -0.54,F1 <= 0.83
4,F5 <= 0.61,F5 <= 0.61
...,...,...
78,background1 <= -0.08,None
79,background1 <= -0.10,None
80,background2 <= -0.08,None
81,background5 > -0.08,None


In [13]:
DG = exp.build_fold_predicate_graph(
    bags_result=folds_result,           # Resultado dos folds (KS + Round-Robin)
    mi_results_dict=mi_results_dict,    # Rankings de Covariância por fold
    predicates_df=predicates_quantiles[0],  # DataFrame com metadados dos predicados
    random_state=42,                    # Semente para reprodutibilidade
    show_details=True,                  # Mostra detalhes da resolução
    normalize_weights=True,            # False = peso inteiro | True = peso [1/k, 1]
    weight_mode='cooccurrence',         # 'ranking' ou 'cooccurrence'
    co_occurrence_matrix=co_occurrence_matrix_df,  # Necessário se weight_mode='cooccurrence'
    apply_confidence_multiplier=True,    # True = peso × score | False = só co-ocorrência, o score de confiança é calculado a partir do ranking do predicado no
    accumulate_cooccurrence_weights=True  # quando True, soma pesos de co-ocorrência entre múltiplas aparições do mesmo predicado
)
DG

# Calcula LRC para cada nó e compõe DataFrame
lrc_ks_df = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
# Seleciona apenas uma ocorrência por zona (maior LRC)
lrc_ks_unique_df = lrc_ks_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_ks_df

 AVISO: normalize_weights=True ignorado em weight_mode='cooccurrence'
   (Pesos de co-ocorrência representam contagens reais de amostras)

CONSTRUÇÃO DO GRAFO DE PREDICADOS
Modo de peso: COOCCURRENCE
Fonte: Matriz de co-ocorrência global
Sub-estratégia: ACUMULATIVA (soma valores da matriz)

Folds processados: 2
Arestas criadas (antes de resolver bidirecionais): 159

RESOLUÇÃO DE ARESTAS BIDIRECIONAIS
Total de pares bidirecionais encontrados: 5
Critério de desempate: PESO ACUMULADO (soma das co-ocorrências locais)

[F1 <= 0.83 ↔ F5 > -0.54]  EMPATE (peso=143.00)
  ✗ Removida (aleatório): F1 <= 0.83 → F5 > -0.54
  ✓ Mantida:  F5 > -0.54 → F1 <= 0.83

[F4 > -0.24 ↔ F6 > -0.42]  EMPATE (peso=104.00)
  ✗ Removida (aleatório): F6 > -0.42 → F4 > -0.24
  ✓ Mantida:  F4 > -0.24 → F6 > -0.42

[F6 > 0.13 ↔ F3 > -0.23]  EMPATE (peso=57.00)
  ✗ Removida (aleatório): F3 > -0.23 → F6 > 0.13
  ✓ Mantida:  F6 > 0.13 → F3 > -0.23

[F4 > -0.47 ↔ F2 > -0.45]  EMPATE (peso=134.00)
  ✗ Removida (aleatório):

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide



,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,F1 > -0.71,3.804956,F1,-0.71,>
1,F5 > -0.54,2.538750,F5,-0.54,>
2,F1 > -0.69,2.481394,F1,-0.69,>
3,F1 <= 0.83,2.466480,F1,0.83,<=
4,F5 <= 0.61,2.442696,F5,0.61,<=
...,...,...,...,...,...
87,background5 <= -0.08,0.447821,background5,-0.08,<=
88,background1 > 0.10,0.302893,background1,0.10,>
89,background3 <= -0.09,0.000090,background3,-0.09,<=
90,Class_A,0.000000,None,None,None


# **Variando o numero de folds - modo cooccurrence**

In [14]:
# Loop para gerar múltiplos grafos e DataFrames LRC para diferentes valores de k_folds

folds_list = [2, 3, 4, 5, 6]  # Exemplo de diferentes valores de k_folds

graphs_ks_by_fold = {}
lrc_ks_df_by_fold = {}
lrc_ks_unique_df_by_fold = {}
all_fold_results_cooc = {}

for k_folds in folds_list:
    print(f"\n{'='*70}")
    print(f"Processando k_folds: {k_folds}")
    print(f"{'='*70}\n")

    # Geração dos folds
    folds_result = ksf.kfold_predicates_roundrobin(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_pred_cont,
        predicates_df=predicates_quantiles[0],
        k_folds=k_folds,
        min_samples_ratio=0.001,
        verbose=True,
        per_predicate=True
    )

    # Adiciona classe prevista (A/B) em cada DataFrame de predicado
    for fold_name, pred_dict in folds_result.items():
        for rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B')

    # Calcula métricas de covariância
    mi_results_dict = exp.calculate_predicate_metrics(
        bags_result=folds_result,
        metric='covariance',
        threshold=0.001,
    )

        # Salvar no dicionário principal
    all_fold_results_cooc[k_folds] = {
        'cov_results_dict': mi_results_dict
    }

    # Constrói o grafo
    DG = exp.build_fold_predicate_graph(
        bags_result=folds_result,
        mi_results_dict=mi_results_dict,
        predicates_df=predicates_quantiles[0],
        random_state=42,
        show_details=True,
        normalize_weights=True,
        weight_mode='cooccurrence',
        co_occurrence_matrix=co_occurrence_matrix_df,
        apply_confidence_multiplier=True,
        accumulate_cooccurrence_weights=True
    )

    # Calcula LRC para cada nó e compõe DataFrame
    lrc_ks_df = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_ks_unique_df = lrc_ks_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)

    # Salva nos dicionários
    graphs_ks_by_fold[k_folds] = DG
    lrc_ks_df_by_fold[k_folds] = lrc_ks_df
    lrc_ks_unique_df_by_fold[k_folds] = lrc_ks_unique_df

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:48:04,517 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:48:04,521 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:48:04,556 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:48:04,559 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 


Processando k_folds: 2

Configuração KS + Round-Robin
Estratégia: PER-PREDICATE (individual)
Total de amostras: 211
Número de folds: 2
Amostras por fold (aprox.): 105
Mínimo de amostras por predicado: 2 (0% do fold)

Iniciando estratégia PER-PREDICATE...
KS será aplicado individualmente para cada predicado.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:48:04,742 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:48:04,751 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 96
Predicados válidos: 96
Predicados eliminados: 0
Folds criados: 2

Estatísticas por predicado válido:
  'F1 <= -0.71': 46 amostras, folds: [23, 23]
  'F1 > -0.71': 165 amostras, folds: [83, 82]
  'F1 <= -0.69': 79 amostras, folds: [40, 39]
  'F1 > -0.69': 132 amostras, folds: [66, 66]
  'F1 <= 0.22': 126 amostras, folds: [63, 63]
  ... e mais 91 predicados

Predicados por fold:
  Fold_1: 96 predicados
  Fold_2: 96 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: normalize_weights=True ignorado em weight_mode='cooccurrence'
   (Pesos de co-ocorrência representam contagens reais de amostras)

CONSTRUÇÃO DO GRAFO DE PREDICADOS
Modo de peso: COOCCURRENCE
Fonte: Matriz de co-ocorrência global
Sub-estratégia: ACUMULATIVA (soma valores da matriz)

Folds processados: 2
Arestas criadas (antes de resolver bidirecionais): 159

RESOLUÇÃO DE ARESTAS BIDIRECIONAIS
Total de pares bidirecionais encontr

...

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:48:16,306 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:48:16,307 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and

 AVISO: normalize_weights=True ignorado em weight_mode='cooccurrence'
   (Pesos de co-ocorrência representam contagens reais de amostras)

CONSTRUÇÃO DO GRAFO DE PREDICADOS
Modo de peso: COOCCURRENCE
Fonte: Matriz de co-ocorrência global
Sub-estratégia: ACUMULATIVA (soma valores da matriz)

Folds processados: 5
Arestas criadas (antes de resolver bidirecionais): 384

RESOLUÇÃO DE ARESTAS BIDIRECIONAIS
Total de pares bidirecionais encontrados: 15
Critério de desempate: PESO ACUMULADO (soma das co-ocorrências locais)

[F1 > -0.71 ↔ F1 > -0.69]
  ✗ Removida: F1 > -0.69 → F1 > -0.71 (peso=132.00, peso=132.00)
  ✓ Mantida:  F1 > -0.71 → F1 > -0.69 (peso=396.00, peso=396.00)

[F1 > -0.69 ↔ F5 > -0.54]
  ✗ Removida: F5 > -0.54 → F1 > -0.69 (peso=92.00, peso=92.00)
  ✓ Mantida:  F1 > -0.69 → F5 > -0.54 (peso=276.00, peso=276.00)

[F5 > -0.54 ↔ F1 <= 0.83]
  ✗ Removida: F1 <= 0.83 → F5 > -0.54 (peso=143.00, peso=143.00)
  ✓ Mantida:  F5 > -0.54 → F1 <= 0.83 (peso=429.00, peso=429.00)

[F2 > -0.4

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:48:16,319 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:48:16,349 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 96
Predicados válidos: 96
Predicados eliminados: 0
Folds criados: 6

Estatísticas por predicado válido:
  'F1 <= -0.71': 46 amostras, folds: [8, 8, 8, 8, 7, 7]
  'F1 > -0.71': 165 amostras, folds: [28, 28, 28, 27, 27, 27]
  'F1 <= -0.69': 79 amostras, folds: [14, 13, 13, 13, 13, 13]
  'F1 > -0.69': 132 amostras, folds: [22, 22, 22, 22, 22, 22]
  'F1 <= 0.22': 126 amostras, folds: [21, 21, 21, 21, 21, 21]
  ... e mais 91 predicados

Predicados por fold:
  Fold_1: 96 predicados
  Fold_2: 96 predicados
  Fold_3: 96 predicados
  Fold_4: 96 predicados
  Fold_5: 96 predicados
  Fold_6: 96 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: normalize_weights=True ignorado em weight_mode='cooccurrence'
   (Pesos de co-ocorrência representam contagens reais de amostras)

CONSTRUÇÃO DO GRAFO DE PREDICADOS
Modo de peso: COOCCURRENCE
Fonte: Matriz de co-ocorrência global
Sub-estratégia: ACUMULATIVA (som

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide



In [15]:
features_importance = pd.DataFrame({
    'Vip' : vip_scores_unique_df['Zone'].iloc[:10].values,
    'Reg_coef' : reg_vet_unique_df['Zone'].iloc[:10].values,
    'Shap' : shap_unique_df['Zone'].iloc[:10].values
    })

for k_folds, lrc_unique_df in lrc_ks_unique_df_by_fold.items():
     features_importance[f'LRC_kfold_{k_folds}'] = lrc_unique_df['Zone'].iloc[:10].values

for seed, lrc_unique_df in lrc_unique_by_seed.items():
    features_importance[f'LRC_Seed_{seed}'] = lrc_unique_df['Zone'].iloc[:10].values


# vamos exportar o df features_importance para um arquivo excel onde vamos nomear a sheet de acordo com as comparacoes feitas
features_importance.to_excel('features_importance_synthetic.xlsx', index=False, sheet_name='KS_fold_cooc')
features_importance

,Vip,Reg_coef,Shap,LRC_kfold_2,LRC_kfold_3,LRC_kfold_4,LRC_kfold_5,LRC_kfold_6,LRC_Seed_0,LRC_Seed_1,LRC_Seed_42
0,F1,F1,F1,F1,F1,F1,F1,F1,F1,F1,F1
1,F5,F5,F4,F5,F3,F5,F5,F5,F5,F5,F5
2,F4,F4,F6,F3,F5,F2,F2,F4,F4,F4,F4
3,F3,F3,F3,F4,F2,F3,F4,F3,F2,F6,F2
4,F6,F6,F5,F2,F6,F6,F6,F2,F6,F2,F3
5,F2,F2,F2,F6,F4,F4,F3,background6,F3,F3,F6
6,background4,background4,background3,background6,background2,background4,background4,F6,background6,background3,background4
7,background5,background5,background4,background5,background4,background3,background3,background1,background3,background5,background6
8,background6,background6,background5,background3,background6,background6,background1,background3,background5,background4,background1
9,background3,background3,background1,background4,background1,background5,background2,background2,background4,background1,background5


In [16]:
# RBO (Rank-Biased Overlap) para comparar rankings
import rbo
rbo_results = {}
reference_list = features_importance['Vip'].tolist()
methods = ['Reg_coef', 'Shap'] + [f'LRC_kfold_{fold}' for fold in folds_list] + [f'LRC_Seed_{seed}' for seed in random_seeds]
for method in methods:
    compare_list = features_importance[method].tolist()
    score = rbo.RankingSimilarity(reference_list, compare_list).rbo(p=0.7, k=10)
    rbo_results[method] = score
rbo_results = pd.DataFrame(list(rbo_results.items()), columns=['Method','RBO_Score'])
rbo_results.insert(0, 'Reference', 'Vip')
rbo_results.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_results.to_excel('rbo_synthetic.xlsx', index=False, sheet_name='KS_fold_cooc')
rbo_results

,Reference,Method,RBO_Score
0,Vip,Reg_coef,0.971752
6,Vip,LRC_kfold_6,0.931460
9,Vip,LRC_Seed_42,0.925401
8,Vip,LRC_Seed_1,0.920359
7,Vip,LRC_Seed_0,0.918481
2,Vip,LRC_kfold_2,0.898294
4,Vip,LRC_kfold_4,0.877612
5,Vip,LRC_kfold_5,0.873269
1,Vip,Shap,0.780765
3,Vip,LRC_kfold_3,0.765148


# **Variando o numero de folds - modo ranking**

In [17]:
# Loop para gerar múltiplos grafos e DataFrames LRC para diferentes valores de k_folds

folds_list = [2, 3, 4, 5, 6]  # Exemplo de diferentes valores de k_folds

graphs_ks_by_fold = {}
lrc_ks_df_by_fold = {}
lrc_ks_unique_df_by_fold = {}
all_fold_results_rank = {}

for k_folds in folds_list:
    print(f"\n{'='*70}")
    print(f"Processando k_folds: {k_folds}")
    print(f"{'='*70}\n")

    # Geração dos folds
    folds_result = ksf.kfold_predicates_roundrobin(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_pred_cont,
        predicates_df=predicates_quantiles[0],
        k_folds=k_folds,
        min_samples_ratio=0.001,
        verbose=True,
        per_predicate=True
    )

    # Adiciona classe prevista (A/B) em cada DataFrame de predicado
    for fold_name, pred_dict in folds_result.items():
        for rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B')

    # Calcula métricas de covariância
    mi_results_dict = exp.calculate_predicate_metrics(
        bags_result=folds_result,
        metric='covariance',
        threshold=0.001,
    )

        # Salvar no dicionário principal
    all_fold_results_rank[k_folds] = {
        'cov_results_dict': mi_results_dict
    }

    # Constrói o grafo
    DG = exp.build_fold_predicate_graph(
        bags_result=folds_result,
        mi_results_dict=mi_results_dict,
        predicates_df=predicates_quantiles[0],
        random_state=42,
        show_details=True,
        normalize_weights=True,
        weight_mode='ranking',
        co_occurrence_matrix=co_occurrence_matrix_df,
        apply_confidence_multiplier=True,
        accumulate_cooccurrence_weights=True
    )

    # Calcula LRC para cada nó e compõe DataFrame
    lrc_ks_df = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_ks_unique_df = lrc_ks_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)

    # Salva nos dicionários
    graphs_ks_by_fold[k_folds] = DG
    lrc_ks_df_by_fold[k_folds] = lrc_ks_df
    lrc_ks_unique_df_by_fold[k_folds] = lrc_ks_unique_df

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:48:23,049 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:48:23,052 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 


Processando k_folds: 2

Configuração KS + Round-Robin
Estratégia: PER-PREDICATE (individual)
Total de amostras: 211
Número de folds: 2
Amostras por fold (aprox.): 105
Mínimo de amostras por predicado: 2 (0% do fold)

Iniciando estratégia PER-PREDICATE...
KS será aplicado individualmente para cada predicado.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:48:23,257 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:48:23,273 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 96
Predicados válidos: 96
Predicados eliminados: 0
Folds criados: 2

Estatísticas por predicado válido:
  'F1 <= -0.71': 46 amostras, folds: [23, 23]
  'F1 > -0.71': 165 amostras, folds: [83, 82]
  'F1 <= -0.69': 79 amostras, folds: [40, 39]
  'F1 > -0.69': 132 amostras, folds: [66, 66]
  'F1 <= 0.22': 126 amostras, folds: [63, 63]
  ... e mais 91 predicados

Predicados por fold:
  Fold_1: 96 predicados
  Fold_2: 96 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: apply_confidence_multiplier=True ignorado em weight_mode='ranking'
   (Multiplicador de confiança só se aplica ao modo 'cooccurrence')
 AVISO: accumulate_cooccurrence_weights=True ignorado em weight_mode='ranking'
   (Acumulação de co-ocorrências só se aplica ao modo 'cooccurrence')

CONSTRUÇÃO DO GRAFO DE PREDICADOS
Modo de peso: RANKING

Folds processados: 2
Arestas criadas (antes de resolver bidirecionais): 159

RESOLUÇÃO DE 

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:48:26,061 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:48:26,065 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 

...


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 96
Predicados válidos: 96
Predicados eliminados: 0
Folds criados: 5

Estatísticas por predicado válido:
  'F1 <= -0.71': 46 amostras, folds: [10, 9, 9, 9, 9]
  'F1 > -0.71': 165 amostras, folds: [33, 33, 33, 33, 33]
  'F1 <= -0.69': 79 amostras, folds: [16, 16, 16, 16, 15]
  'F1 > -0.69': 132 amostras, folds: [27, 27, 26, 26, 26]
  'F1 <= 0.22': 126 amostras, folds: [26, 25, 25, 25, 25]
  ... e mais 91 predicados

Predicados por fold:
  Fold_1: 96 predicados
  Fold_2: 96 predicados
  Fold_3: 96 predicados
  Fold_4: 96 predicados
  Fold_5: 96 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001


C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:48:36,613 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:48:36,615 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 

 AVISO: apply_confidence_multiplier=True ignorado em weight_mode='ranking'
   (Multiplicador de confiança só se aplica ao modo 'cooccurrence')
 AVISO: accumulate_cooccurrence_weights=True ignorado em weight_mode='ranking'
   (Acumulação de co-ocorrências só se aplica ao modo 'cooccurrence')

CONSTRUÇÃO DO GRAFO DE PREDICADOS
Modo de peso: RANKING

Folds processados: 5
Arestas criadas (antes de resolver bidirecionais): 384

RESOLUÇÃO DE ARESTAS BIDIRECIONAIS
Total de pares bidirecionais encontrados: 15
Critério de desempate: PESO ACUMULADO (soma dos ranks invertidos)

[F1 > -0.71 ↔ F1 > -0.69]
  ✗ Removida: F1 > -0.69 → F1 > -0.71 (peso=1.00, peso=1.00)
  ✓ Mantida:  F1 > -0.71 → F1 > -0.69 (peso=3.00, peso=3.00)

[F1 > -0.69 ↔ F5 > -0.54]
  ✗ Removida: F5 > -0.54 → F1 > -0.69 (peso=0.99, peso=0.99)
  ✓ Mantida:  F1 > -0.69 → F5 > -0.54 (peso=2.96, peso=2.96)

[F5 > -0.54 ↔ F1 <= 0.83]
  ✗ Removida: F1 <= 0.83 → F5 > -0.54 (peso=0.98, peso=0.98)
  ✓ Mantida:  F5 > -0.54 → F1 <= 0.83 (pe

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:48:36,673 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-20 11:48:36,675 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 96
Predicados válidos: 96
Predicados eliminados: 0
Folds criados: 6

Estatísticas por predicado válido:
  'F1 <= -0.71': 46 amostras, folds: [8, 8, 8, 8, 7, 7]
  'F1 > -0.71': 165 amostras, folds: [28, 28, 28, 27, 27, 27]
  'F1 <= -0.69': 79 amostras, folds: [14, 13, 13, 13, 13, 13]
  'F1 > -0.69': 132 amostras, folds: [22, 22, 22, 22, 22, 22]
  'F1 <= 0.22': 126 amostras, folds: [21, 21, 21, 21, 21, 21]
  ... e mais 91 predicados

Predicados por fold:
  Fold_1: 96 predicados
  Fold_2: 96 predicados
  Fold_3: 96 predicados
  Fold_4: 96 predicados
  Fold_5: 96 predicados
  Fold_6: 96 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: apply_confidence_multiplier=True ignorado em weight_mode='ranking'
   (Multiplicador de confiança só se aplica ao modo 'cooccurrence')
 AVISO: accumulate_cooccurrence_weights=True ignorado em weight_mode='ranking'
   (Acumulação de co-ocorrências só se aplica ao

In [18]:
features_importance = pd.DataFrame({
    'Vip' : vip_scores_unique_df['Zone'].iloc[:10].values,
    'Reg_coef' : reg_vet_unique_df['Zone'].iloc[:10].values,
    'Shap' : shap_unique_df['Zone'].iloc[:10].values
    })

for k_folds, lrc_unique_df in lrc_ks_unique_df_by_fold.items():
    features_importance[f'LRC_kfold_{k_folds}'] = lrc_unique_df['Zone'].iloc[:10].values

for seed, lrc_unique_df in lrc_unique_by_seed.items():
    features_importance[f'LRC_Seed_{seed}'] = lrc_unique_df['Zone'].iloc[:10].values
    
with pd.ExcelWriter('features_importance_synthetic.xlsx', mode='a', engine='openpyxl') as writer:
    features_importance.to_excel(writer, sheet_name='KS_fold_rank', index=False)
features_importance

,Vip,Reg_coef,Shap,LRC_kfold_2,LRC_kfold_3,LRC_kfold_4,LRC_kfold_5,LRC_kfold_6,LRC_Seed_0,LRC_Seed_1,LRC_Seed_42
0,F1,F1,F1,F1,F3,F1,F1,F1,F1,F1,F1
1,F5,F5,F4,F5,F1,F5,F5,F4,F5,F5,F5
2,F4,F4,F6,F4,F6,F2,F2,F3,F4,F4,F4
3,F3,F3,F3,F3,F5,F6,F6,F5,F2,F6,F2
4,F6,F6,F5,F2,F2,F4,F4,F6,F6,F2,F3
5,F2,F2,F2,F6,F4,F3,F3,F2,F3,F3,F6
6,background4,background4,background3,background4,background4,background4,background3,background6,background6,background3,background4
7,background5,background5,background4,background3,background2,background5,background2,background2,background3,background5,background6
8,background6,background6,background5,background5,background1,background3,background4,background5,background5,background4,background1
9,background3,background3,background1,background1,background6,background6,background1,background1,background4,background1,background5


In [19]:
# RBO (Rank-Biased Overlap) para comparar rankings
import rbo
rbo_results = {}
reference_list = features_importance['Vip'].tolist()
methods = ['Reg_coef', 'Shap'] + [f'LRC_kfold_{fold}' for fold in folds_list] + [f'LRC_Seed_{seed}' for seed in random_seeds]
for method in methods:
    compare_list = features_importance[method].tolist()
    score = rbo.RankingSimilarity(reference_list, compare_list).rbo(p=0.7, k=10)
    rbo_results[method] = score
rbo_results = pd.DataFrame(list(rbo_results.items()), columns=['Method','RBO_Score'])
rbo_results.insert(0, 'Reference', 'Vip')
rbo_results.sort_values(by='RBO_Score', ascending=False, inplace=True)

# Supondo que features_importance já está criado
with pd.ExcelWriter('rbo_synthetic.xlsx', mode='a', engine='openpyxl') as writer:
    rbo_results.to_excel(writer, sheet_name='KS_fold_rank', index=False)
rbo_results

,Reference,Method,RBO_Score
0,Vip,Reg_coef,0.971752
2,Vip,LRC_kfold_2,0.951126
9,Vip,LRC_Seed_42,0.925401
8,Vip,LRC_Seed_1,0.920359
7,Vip,LRC_Seed_0,0.918481
4,Vip,LRC_kfold_4,0.854975
5,Vip,LRC_kfold_5,0.839413
6,Vip,LRC_kfold_6,0.802191
1,Vip,Shap,0.780765
3,Vip,LRC_kfold_3,0.419269
